# Maths LangChain Toolbox Experiment

This notebook is separate from `maths.ipynb`. It tests a LangChain-style toolbox loop with the local Qwen instruct model.

Flow: route question -> optional calculator tool -> final option letter.

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf latex2sympy2 sympy langchain langchain-core langchain-huggingface

In [ ]:
import os, sys, re, time, json, math, getpass
from fractions import Fraction

import torch
import sympy as sp
from sympy import symbols, simplify, N

BASE_DIR = '/content/gdrive/MyDrive/NLP_assignment'
if os.path.exists(BASE_DIR):
    sys.path.append(BASE_DIR)
else:
    sys.path.append('.')

print('Environment ready')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool

MODEL_ID = 'Qwen/Qwen2.5-Math-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    torch_dtype=torch.float16
)
model.eval()
print(f'Model loaded: {MODEL_ID}')

In [ ]:
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor

TRANSFORMS = standard_transformations + (implicit_multiplication_application, convert_xor)
SYM_LOCAL = {
    'x': sp.symbols('x'), 'y': sp.symbols('y'), 'a': sp.symbols('a'),
    'b': sp.symbols('b'), 'k': sp.symbols('k'), 'lam': sp.symbols('lam'),
    'pi': sp.pi, 'E': sp.E, 'e': sp.E,
}

def parse_math_expr(expr, local_dict=None):
    local = dict(SYM_LOCAL)
    if local_dict:
        local.update(local_dict)
    return parse_expr(clean_math(expr), transformations=TRANSFORMS, local_dict=local)

def qwen_chat(messages, max_new_tokens=160, max_time=None):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]
    kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    if max_time is not None:
        kwargs['max_time'] = max_time
    with torch.no_grad():
        outputs = model.generate(**kwargs)
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

def options_text(question):
    return '\n'.join(f'{chr(65+i)}) {opt.text}' for i, opt in enumerate(question.options))

def pick_option_by_letter(question, letter, method):
    idx = ['A', 'B', 'C', 'D'].index(letter)
    return question.options[idx].id, letter, method

def extract_letter(raw):
    boxed = re.findall(r'\\boxed\{([A-D])\}', raw, re.IGNORECASE)
    if boxed:
        return boxed[-1].upper(), 'boxed'
    m = re.search(r'(?:FINAL ANSWER|answer|option|choice)\s*:?\s*([A-D])\b', raw, re.IGNORECASE)
    if m:
        return m.group(1).upper(), 'text'
    letters = re.findall(r'\b([A-D])\b', raw.upper())
    return (letters[-1], 'fallback') if letters else ('C', 'fallback')

def clean_math(s):
    return (s.replace('^', '**')
             .replace('−', '-')
             .replace('–', '-')
             .replace('π', 'pi'))

In [ ]:
@tool
def sympy_solver(problem: str, options: str = '') -> str:
    """Solve algebra, calculus, equations, derivatives, integrals, limits, and optimization using SymPy."""
    x, y, a, b, k = sp.symbols('x y a b k')
    text = clean_math(problem)
    low = text.lower()
    notes = []

    try:
        # Perpendicular lines: Ax + By + C = 0 has slope -A/B.
        if 'perpendicular' in low and 'ax + 2y' in low:
            # x + 2y + 3 has slope -1/2. ax + 2y + 3 has slope -a/2.
            sol = sp.solve(sp.Eq((-sp.Rational(1, 2)) * (-a / 2), -1), a)
            return f'Perpendicular slopes multiply to -1. a = {sol}. Match this value to the options.'

        # Positive difference between roots of a quadratic equation.
        if 'positive difference' in low and 'solutions' in low:
            m = re.search(r'\$([^$]+)=([^$]+)\$', text)
            if m:
                lhs = parse_math_expr(m.group(1), {'x': x})
                rhs = parse_math_expr(m.group(2), {'x': x})
                roots = sp.solve(sp.Eq(lhs, rhs), x)
                diff = abs(sp.simplify(roots[0] - roots[1])) if len(roots) == 2 else roots
                return f'Equation roots: {roots}. Positive difference: {diff}.'

        # Complete the square: ax^2 + bx + c = a(x-h)^2 + k.
        if 'form' in low and '(x - h)^2 + k' in low:
            m = re.search(r'\$([^$]+)\$', text)
            if m:
                expr = parse_math_expr(m.group(1), {'x': x})
                poly = sp.Poly(expr, x)
                A, B, C = poly.all_coeffs()
                h_val = -B / (2 * A)
                k_val = sp.simplify(expr.subs(x, h_val))
                return f'Completed-square vertex value k = {k_val}.'

        # h(4x-1)=2x+7, solve h(x)=x.
        if 'h(4x-1)' in low and 'h(x) = x' in low:
            u = sp.symbols('u')
            h_u = sp.simplify(2 * ((u + 1) / 4) + 7)
            sol = sp.solve(sp.Eq(h_u, u), u)
            return f'h(u) = {h_u}. h(x)=x gives x = {sol}.'

        # Tangency y = mx to y = e^(b x): e^(b x0)=m*x0 and b*e^(b x0)=m -> x0=1/b, b=m/e.
        if 'tangent' in low and 'e**(bx)' in low:
            m = re.search(r'y\s*=\s*([0-9]+)x', text)
            slope = sp.Integer(m.group(1)) if m else sp.Integer(10)
            return f'Tangency gives x0=1/b and e^(b*x0)=slope*x0, so b = slope/e = {slope}/E.'

        # Two random points cut a segment into 3 parts; triangle probability is 1/4.
        if 'divide the segment into three' in low and 'triangle' in low:
            return 'Classic broken-stick result: probability the three pieces form a triangle is 1/4.'

        # Generic derivative/integral/limit from first LaTeX block.
        if any(w in low for w in ['derivative', 'differentiate', 'integral', 'limit']):
            from latex2sympy2 import latex2sympy
            blocks = re.findall(r'\$([^$]+)\$', problem)
            if blocks:
                expr = latex2sympy(max(blocks, key=len))
                if 'integral' in low:
                    return f'Integral: {sp.integrate(expr, x)}.'
                if 'limit' in low:
                    return f'Expression parsed as {expr}. Need limit point from question/options.'
                return f'Derivative: {sp.diff(expr, x)}.'

        return 'No confident SymPy shortcut found.'
    except Exception as e:
        return f'SymPy tool failed: {type(e).__name__}: {e}'

@tool
def stats_probability_solver(problem: str, options: str = '') -> str:
    """Solve common probability and statistics calculations: Bayes, binomial, normal quartiles, draws, expected value, r^2."""
    text = problem.lower().replace(',', '')
    try:
        if 'correlation of 0.6' in text and 'correlation of 0.3' in text:
            return 'Explained variation is r^2. Ratio = 0.6^2 / 0.3^2 = 4.'

        if 'first quartile' in text and 'normally distributed' in text:
            nums = [float(n) for n in re.findall(r'\$?([0-9]+(?:\.[0-9]+)?)', text)]
            mean, q1 = nums[0], nums[1]
            sigma = abs(mean - q1) / 0.6745
            return f'For a normal distribution, Q1 = mean - 0.6745*sigma. sigma = {sigma:.3f}.'

        if 'five children' in text and 'at least half' in text:
            p = sum(math.comb(5, r) for r in range(3, 6)) / 2**5
            return f'P(at least 3 girls out of 5) = {Fraction(p).limit_denominator()} = {p:.4f}.'

        if '4 red and 6 blue marbles' in text and 'same color' in text:
            p = (4/10)*(3/9) + (6/10)*(5/9)
            return f'P(same color) = 4/10*3/9 + 6/10*5/9 = {Fraction(p).limit_denominator()}.'

        if '4 white balls and 4 black balls' in text and 'alternate colors' in text:
            p = 2 * math.factorial(4) * math.factorial(4) / math.factorial(8)
            return f'Only two alternating color patterns. Probability = 2*4!*4!/8! = {Fraction(p).limit_denominator()}.'

        if 'active chip' in text and 'alarm sounds' in text:
            prior = 0.005
            hit = 0.98
            false_alarm = 0.03
            post = hit * prior / (hit * prior + false_alarm * (1 - prior))
            return f'Bayes: P(active|alarm) = {post:.6f} ({post*100:.2f}%).'

        if 'expected return' in text or 'maximize expected' in text:
            return 'Compute each option by sum(probability * return). Pick the largest expected value, unless the option text asks for risk/threshold guarantees.'

        return 'No confident stats/probability shortcut found.'
    except Exception as e:
        return f'Stats tool failed: {type(e).__name__}: {e}'

@tool
def linear_algebra_solver(problem: str, options: str = '') -> str:
    """Solve matrix, eigenvalue, trace, determinant, vector norm, and characteristic-polynomial questions."""
    lam = sp.symbols('lambda')
    text = clean_math(problem)
    low = text.lower()
    try:
        if 'det(a' in low and 'lambda' in low:
            poly_match = re.search(r'=\s*([^,]+)', text)
            if poly_match:
                expr_text = poly_match.group(1).replace('lambda', 'lam')
                lam_sym = sp.symbols('lam')
                poly = parse_math_expr(expr_text, {'lam': lam_sym})
                roots = sp.solve(sp.Eq(poly, 0), lam_sym)
                det_a = sp.simplify(poly.subs(lam_sym, 0))
                trace_a = sp.simplify(sum(roots)) if roots else 'unknown'
                return f'Characteristic polynomial roots/eigenvalues: {roots}. trace={trace_a}. det(A)=p(0)={det_a}.'

        if 'a = a^(-1)' in low and '2 x 2' in low:
            return 'A=A^-1 means eigenvalues are +/-1. Since A is not I or -I, the 2x2 real case has eigenvalues 1 and -1, so trace is 0.'

        if 'not always true' in low and 'r^k' in low:
            return 'Cauchy-Schwarz, norm zero iff zero, and nonnegativity are always true. Equality |x+y|=|x|+|y| is not always true.'

        if 'norm' in low and 'inner product' in low:
            return 'Lp norm comes from an inner product only for p=2, by the parallelogram law.'

        return 'No confident linear algebra shortcut found.'
    except Exception as e:
        return f'Linear algebra tool failed: {type(e).__name__}: {e}'

TOOLS = {
    'sympy_solver': sympy_solver,
    'stats_probability_solver': stats_probability_solver,
    'linear_algebra_solver': linear_algebra_solver,
}
print('Tools ready:', ', '.join(TOOLS))

In [ ]:
router_prompt = PromptTemplate.from_template('''You are routing a multiple-choice math question to a calculator tool.
Available tools:
- sympy_solver: equations, algebra, calculus, derivative, integral, limit, optimization.
- stats_probability_solver: probability, statistics, Bayes, binomial, normal distribution, expected value.
- linear_algebra_solver: matrices, eigenvalues, trace, determinant, vector norms.
- none: conceptual question or no useful calculator.

Return JSON only, exactly like: {{"tool":"sympy_solver"}}

Question:
{question}

Options:
{options}
''')

final_prompt = PromptTemplate.from_template('''You are a fast math multiple-choice solver.
Use the calculator result if it is relevant. Be brief.
Reply with the final option letter inside one box, like \\boxed{{A}}.

Question:
{question}

Options:
{options}

Calculator result:
{tool_result}
''')

direct_prompt = PromptTemplate.from_template('''You are a fast math multiple-choice solver.
Reason briefly. Reply with the final option letter inside one box, like \\boxed{{A}}.

Question:
{question}

Options:
{options}
''')

def parse_router(raw):
    m = re.search(r'\{.*?\}', raw, re.S)
    if not m:
        return 'none'
    try:
        tool_name = json.loads(m.group(0)).get('tool', 'none')
        return tool_name if tool_name in TOOLS else 'none'
    except Exception:
        return 'none'

def choose_answer_langchain_toolbox(question, max_seconds=28.5):
    opts = options_text(question)
    started = time.time()

    route_text = router_prompt.format(question=question.text, options=opts)
    route_raw = qwen_chat([
        {'role': 'system', 'content': 'Return JSON only.'},
        {'role': 'user', 'content': route_text},
    ], max_new_tokens=50, max_time=4.0)
    tool_name = parse_router(route_raw)
    print(f'[Router] raw={route_raw.strip()} | selected={tool_name}')

    tool_result = 'No tool used.'
    if tool_name != 'none':
        tool_result = TOOLS[tool_name].invoke({'problem': question.text, 'options': opts})
    print(f'[Tool] {tool_name}: {tool_result}')

    remaining = max(1.0, max_seconds - (time.time() - started))
    if tool_name == 'none':
        answer_text = direct_prompt.format(question=question.text, options=opts)
    else:
        answer_text = final_prompt.format(question=question.text, options=opts, tool_result=tool_result)

    raw = qwen_chat([
        {'role': 'system', 'content': 'Answer with one boxed option letter.'},
        {'role': 'user', 'content': answer_text},
    ], max_new_tokens=140, max_time=remaining)
    letter, rule = extract_letter(raw)
    wrapup_used = False

    # If the final answer was not clear, use the last few tokens for a forced guess.
    remaining = max_seconds - (time.time() - started)
    if rule == 'fallback' and remaining > 0.7:
        wrap_raw = qwen_chat([
            {'role': 'system', 'content': 'Reply only with one boxed option letter.'},
            {'role': 'user', 'content': f"Time is almost finished. Submit the nearest guess now. Options are A, B, C, D. Reply only as \\boxed{{A}}, \\boxed{{B}}, \\boxed{{C}}, or \\boxed{{D}}.\n\nPrevious answer:\n{raw[-900:]}"},
        ], max_new_tokens=20, max_time=remaining)
        wrap_letter, wrap_rule = extract_letter(wrap_raw)
        raw += '\n' + wrap_raw
        letter, rule = wrap_letter, 'wrapup_' + wrap_rule
        wrapup_used = True
        print(f'[Wrapup Raw] {wrap_raw.strip()}')

    elapsed = time.time() - started
    print(f'[Final Raw]\n{raw.strip()}')
    print(f'[Extract] letter={letter} | rule={rule} | wrapup={wrapup_used} | elapsed={elapsed:.2f}s')
    return pick_option_by_letter(question, letter, f'langchain_{tool_name}_{rule}')

In [ ]:
# Quick local smoke test without the game API.
class Opt:
    def __init__(self, text, id=None):
        self.text = text
        self.id = id or text

class FakeQuestion:
    def __init__(self, text, options):
        self.text = text
        self.options = [Opt(t, i) for i, t in enumerate(options)]

fake = FakeQuestion(
    'The graph of the equation $x + 2y + 3 = 0$ is perpendicular to the graph of the equation $ax + 2y + 3 = 0$. What is the value of $a$?',
    ['-4', '-1', '1', '4']
)
# choose_answer_langchain_toolbox(fake)

In [ ]:
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

USERNAME = input('username: ')
PASSWORD = getpass.getpass('password: ')

client = MillionaireClient('http://131.175.15.22:51111/')
user = client.login(USERNAME, PASSWORD)
print(f'Logged in as: {user.username}')

competitions = client.competitions.list_all()
COMPETITION_ID = competitions[3].id
game = client.game.start(competition_id=COMPETITION_ID, mode='text')

while game.in_progress:
    question = game.current_question
    if question is None:
        break

    print(f"\n{'='*60}\nLevel: {game.current_level}\nQ: {question.text}")
    for i, opt in enumerate(question.options):
        print(f'  {chr(65+i)}) {opt.text}')

    t0 = time.time()
    option_id, letter, method = choose_answer_langchain_toolbox(question)
    t1 = time.time()
    print(f'Predicted: {letter} | Method: {method} | Time taken: {t1-t0:.2f}s')

    try:
        result = game.answer(option_id)
        print(f'Correct: {result.correct} | Earned: {result.earned_amount}')
    except TimeoutError:
        print('Timed out')
        break
    except RateLimitError:
        print('Rate limited, waiting 5s')
        time.sleep(5)
        result = game.answer(option_id)
        print(f'Correct: {result.correct} | Earned: {result.earned_amount}')

    if result.game_over:
        break
    time.sleep(1)

print(f"\n{'='*60}\nGame over. Final score: {game.earned_amount}")